# **Splitting Agent Testing**

## **Boilerplate**

In [23]:
from dotenv import load_dotenv
import os

load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("OpenAI api key set")
else:
    raise ValueError("OpenAI api key is not set")

OpenAI api key set


In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4")

## **Graph Schema**

In [25]:
from pydantic import BaseModel, Field
from typing import List, Annotated
import operator

class graph_schema(BaseModel):
    measures: List = Field(description="a list of the measures to be analyzed")
    parts: List = Field(description="a list of the different parts of the song split up")

## **LLM Schema**

In [26]:
class Note(BaseModel):
    string: int
    fret: int
    midi: int

class Beat(BaseModel):
    time: float
    duration: float
    notes: List[Note]

class Measure(BaseModel):
    index: int
    beats: List[Beat]

class Part(BaseModel):
    section_type: str = Field(description="e.g. intro, verse, chorus, bridge, guitar solo, outro, break")
    measures: List[Measure] = Field(description="the measures belonging to this section, same format as the input guitar tab")

class llm_schema(BaseModel):
    parts: List[Part] = Field(description="a list of the different parts of the song split up")

llm_with_schema = llm.with_structured_output(llm_schema)

## **Node Functions**

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage, HumanMessage
import json

def split_song(state: graph_schema) -> graph_schema:
    
    measures = ", ".join(json.dumps(measure) for measure in state.measures)

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a guitar music analyst that looks at the guitar tab of a song and predicts the cutoffs between its different sections (ex. verse, chorus, bridge, guitar solo)"),
            ("human", "You are a music analysis expert specializing in guitar compositions. Your task is to analyze a guitar tab in JSON format and identify its distinct musical sections."
            "## Input Format"
            "The guitar tab is provided as a JSON array where each element represents a beat/section with:"
            "- beats: array of individual note events"
            "- time: absolute time in seconds"
            "- notes: array with fret, MIDI value, string number, duration"
            "- index: sequential section number"

            "## Section Types to Identify"
            "Look for common song structures and identify these sections (if present):"
            "- **Intro**: Opening instrumental passage, establishes the key/mood"
            "- **Verse**: Main lyrical/melodic section, typically repeats with variations"
            "- **Chorus**: Memorable hook section, usually higher energy or different melodic character"
            "- **Bridge**: Contrasting section that breaks up verse/chorus pattern"
            "- **Guitar Solo**: Improvisational or lead guitar passage, may have different rhythmic patterns"
            "- **Outro**: Closing section, often simplified or repetitive"
            "- **Break**: Short transitional or rhythmic interlude"

            "## Analysis Guidelines"
            "Consider these elements when identifying sections:"
            "1. **Rhythmic Pattern Changes**: Look for shifts in note density, duration patterns, or timing"
            "2. **Pitch Range**: Identify if melodies move to different string ranges (higher/lower strings)"
            "3. **Motif Repetition**: Same note sequences or patterns indicate section cohesion"
            "4. **Structural Logic**: Real songs follow recognizable patterns (verse-chorus-verse-chorus-bridge, etc.)"
            "5. **Complexity**: Solo sections often have more notes and faster rhythms; verses may be more sparse"

            "## Output Format"
            "Analyze the tab and provide your section splits using this schema:"
            "[{{\"section_type\": \"chorus\", \"measures\": [same format as guitar tab you are analyzing]}},...]"

            "Return ONLY valid JSON matching the output schema above."
            "Here is the guitar tab: {measures}")
        ]
    )    
    chain = prompt | llm_with_schema

    split = chain.invoke({"measures": measures})

    state.parts = split.parts

    return state
    


## **Build Graph**

In [28]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

graph.add_node("split_song", split_song)

graph.add_edge(START, "split_song")
graph.add_edge("split_song", END)

test_graph = graph.compile()

## **Invoke Graph**

In [ ]:
with open("blackbird.txt", "r", encoding="utf-8") as file:
    blackbird=json.loads(file.read())

result = test_graph.invoke({"measures": blackbird, "parts": []})
result

{'measures': [{'beats': [], 'index': 0},
  {'beats': [], 'index': 1},
  {'beats': [{'time': 4.8,
     'notes': [{'fret': 5, 'midi': 50, 'string': 5}],
     'duration': 0.15},
    {'time': 4.95,
     'notes': [{'fret': 7, 'midi': 52, 'string': 5}],
     'duration': 0.3},
    {'time': 5.25,
     'notes': [{'fret': 5, 'midi': 50, 'string': 5}],
     'duration': 0.15},
    {'time': 5.4,
     'notes': [{'fret': 7, 'midi': 52, 'string': 5}],
     'duration': 0.3},
    {'time': 5.7,
     'notes': [{'fret': 5, 'midi': 50, 'string': 5}],
     'duration': 0.15},
    {'time': 5.85,
     'notes': [{'fret': 7, 'midi': 52, 'string': 5}],
     'duration': 0.3},
    {'time': 6.15,
     'notes': [{'fret': 5, 'midi': 50, 'string': 5}],
     'duration': 0.15},
    {'time': 6.3,
     'notes': [{'fret': 7, 'midi': 52, 'string': 5}],
     'duration': 0.3},
    {'time': 6.6,
     'notes': [{'fret': 5, 'midi': 50, 'string': 5}],
     'duration': 0.15},
    {'time': 6.75,
     'notes': [{'fret': 7, 'midi': 52,

## **Push Parts to Track**

In [ ]:
import requests

track_id = "a8aa9dce-35fd-4177-9141-e682ac888976"  # <-- paste the track's uuid here (tracks.id, not track_index)

# `result["parts"]` is a list of Part objects. model_dump() recursively
# converts each one (and its nested Measure/Beat/Note models) into plain
# dicts matching the exact shape the frontend's Track.parts type expects.
def to_plain(obj):
    return obj.model_dump() if isinstance(obj, BaseModel) else obj

parts_payload = [to_plain(part) for part in result["parts"]]

resp = requests.post(
    f"http://localhost:8000/tracks/{track_id}/parts",
    json=parts_payload,
)
resp.raise_for_status()
resp.json()

{'id': '4ea537f4-c472-4bb2-9124-f6c2a5c3bc7f',
 'song_id': '6d70a383-0538-41aa-a0c8-ecd057cc6408',
 'track_index': 0,
 'name': 'guitar',
 'tuning': [64, 59, 55, 50, 45, 40],
 'note_data': [{'beats': [{'time': 0,
     'notes': [{'fret': 0, 'midi': 40, 'string': 6}],
     'duration': 0.3},
    {'time': 0.3,
     'notes': [{'fret': 3, 'midi': 43, 'string': 6}],
     'duration': 0.3},
    {'time': 0.6,
     'notes': [{'fret': 0, 'midi': 45, 'string': 5}],
     'duration': 0.3},
    {'time': 0.9,
     'notes': [{'fret': 2, 'midi': 47, 'string': 5}],
     'duration': 0.3},
    {'time': 1.2,
     'notes': [{'fret': 0, 'midi': 50, 'string': 4}],
     'duration': 0.3},
    {'time': 1.5,
     'notes': [{'fret': 2, 'midi': 52, 'string': 4}],
     'duration': 0.3},
    {'time': 1.8,
     'notes': [{'fret': 0, 'midi': 55, 'string': 3}],
     'duration': 0.3},
    {'time': 2.1,
     'notes': [{'fret': 2, 'midi': 57, 'string': 3}],
     'duration': 0.3}],
   'index': 0},
  {'beats': [{'time': 2.4,
  